## Summary

You've successfully built an HR RAG (Retrieval-Augmented Generation) system! 

### System Components:
- **Document Loaders**: Load PDF and DOCX files from the data directory
- **Text Splitter**: Split documents into 1000-character chunks with 200-character overlap
- **Embeddings**: Use SentenceTransformer for document representation
- **Vector Store**: FAISS for efficient similarity search
- **LLM**: Google Gemini 2.5 Flash for generating responses
- **Memory**: InMemoryChatMessageHistory for maintaining conversation context

### Next Steps:
1. Deploy the Streamlit app: `streamlit run app.py`
2. Add more HR documents to the `data/` folder
3. Test with your actual HR-related documents
4. Customize the system prompt for specific use cases

For production deployment, consider:
- Using persistent vector stores (FAISS index files)
- Implementing proper session management
- Adding document management interfaces
- Monitoring and logging query performance

In [ ]:
# Test the RAG system with sample queries
print("Testing RAG System...\n")

# Sample HR questions
sample_questions = [
    "What are the company leave policies?",
    "How do I apply for vacation?",
    "What benefits does the company provide?"
]

# Test each question
for i, question in enumerate(sample_questions, 1):
    print(f"Question {i}: {question}")
    try:
        response = rag_with_memory.invoke(
            {"question": question},
            config={"configurable": {"session_id": "test_session"}}
        )
        print(f"Answer: {response}\n")
    except Exception as e:
        print(f"Error processing question: {str(e)}\n")

## Section 8: Test the RAG System with Sample Queries

In [ ]:
# Set up chat memory management
store = {}

def get_chat_history(session_id: str):
    """Get or create chat history for a session"""
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# Wrap RAG chain with message history
rag_with_memory = RunnableWithMessageHistory(
    rag_chain,
    get_chat_history,
    input_messages_key="question",
    history_messages_key="chat_history",
)

print("✓ Chat memory management configured")

## Section 7: Implement Chat Memory Management

In [ ]:
# Create prompt template for HR assistant
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are an HR assistant. Answer only from the provided context. If the answer is not in the context, politely say you don't have that information."
    ),
    MessagesPlaceholder(variable_name="chat_history"),
    (
        "human",
        "Context:\n{context}\n\nQuestion: {question}"
    )
])

# Function to format retrieved documents
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Build the RAG chain
print("Building RAG chain...")
rag_chain = (
    RunnablePassthrough.assign(
        context=lambda x: format_docs(retriever.invoke(x["question"]))
    )
    | prompt
    | llm
    | StrOutputParser()
)
print("✓ RAG chain created successfully")

## Section 6: Build RAG Chain with Prompts

In [ ]:
# Initialize Google Gemini LLM
print("Initializing LLM...")
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
    google_api_key=google_api_key
)
print("✓ Gemini 2.5 Flash model initialized")

## Section 5: Initialize LLM

In [ ]:
# Split documents into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

print("Splitting documents into chunks...")
split_docs = text_splitter.split_documents(documents)
print(f"✓ Created {len(split_docs)} document chunks")

# Create embeddings
print("\nCreating embeddings...")
embeddings = SentenceTransformerEmbeddings(
    model_name="all-MiniLM-L6-v2"
)
print("✓ Embeddings model initialized")

# Create FAISS vector store
print("\nCreating FAISS vector store...")
vectorstore = FAISS.from_documents(
    documents=split_docs,
    embedding=embeddings
)
print(f"✓ Vector store created with {len(split_docs)} vectors")

# Set up retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print("✓ Retriever configured to fetch top 3 relevant documents")

## Section 4: Split Documents and Create Vector Store

In [ ]:
# Load PDF and DOCX files from the data directory
pdf_loader = DirectoryLoader(
    path="data",
    glob="**/*.pdf",
    loader_cls=PyPDFLoader
)

docx_loader = DirectoryLoader(
    path="data",
    glob="**/*.docx",
    loader_cls=Docx2txtLoader
)

# Load all documents
print("Loading documents...")
pdf_documents = pdf_loader.load()
docx_documents = docx_loader.load()

# Combine all documents
documents = pdf_documents + docx_documents
print(f"✓ Loaded {len(pdf_documents)} PDF documents and {len(docx_documents)} DOCX documents")
print(f"✓ Total documents: {len(documents)}")

# Display sample document
if documents:
    print(f"\nSample document metadata:")
    print(f"  Source: {documents[0].metadata.get('source', 'N/A')}")
    print(f"  Content length: {len(documents[0].page_content)} characters")

## Section 3: Load and Process Documents

In [ ]:
# Set your Google Gemini API Key
# Replace 'YOUR_API_KEY' with your actual API key from https://makersuite.google.com/app/apikey
google_api_key = "YOUR_API_KEY"

# Verify API key is set
if google_api_key == "YOUR_API_KEY":
    print("⚠️  WARNING: Please set your Google Gemini API Key")
else:
    print("✓ API Key configured")

## Section 2: Configure API Keys

In [ ]:
import os
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

print("✓ All libraries imported successfully")

## Section 1: Import Required Libraries

# HR RAG Chatbot - Jupyter Notebook
## Building a Retrieval-Augmented Generation System for HR Documents

This notebook demonstrates building an intelligent HR assistant using LangChain, FAISS vector stores, and Google Gemini API.